# 15 Build Terrain Features for Puerto Rico

This notebook is the guided Terrain Feature Pack v1 workflow for Puerto Rico.

Use it in order:
1. inspect which sources were discovered locally
2. review the manual follow-up actions that still matter
3. place any needed downloads under `data/staging/terrain/`
4. run the terrain stage when you are ready
5. preview the outputs before deciding whether to ship


In [ ]:
# Cell 1: Setup
import json
import sys
from pathlib import Path


def find_repo_root():
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "README.md").exists() and (candidate / "JupyterNotebooks").exists():
            return candidate
    return p


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.terrain.terrain_feature_pack import inspect_terrain_inputs, run_terrain_feature_pack

try:
    from IPython.display import display
except ImportError:
    display = print

print(f"Repo root: {REPO_ROOT}")


In [ ]:
# Cell 2: Inspect local terrain inputs before running the stage
inventory = inspect_terrain_inputs(repo_root=REPO_ROOT)
print(json.dumps(inventory, indent=2))


In [ ]:
# Cell 3: Review discovered sources and manual follow-up actions
import pandas as pd

source_rows = []
for source_name, details in inventory["sources"].items():
    source_rows.append(
        {
            "source": source_name,
            "status": details.get("status"),
            "paths": "; ".join(details.get("paths", [])),
        }
    )

display(pd.DataFrame(source_rows))

if inventory["manual_actions"]:
    print("Manual follow-up actions:")
    for item in inventory["manual_actions"]:
        print(f"- {item}")
else:
    print("No blocking manual follow-up actions were generated by the inspection step.")


In [ ]:
# Cell 4: Run the terrain stage when ready
RUN_TERRAIN_STAGE = False

if RUN_TERRAIN_STAGE:
    metadata = run_terrain_feature_pack(repo_root=REPO_ROOT)
    print(json.dumps(metadata, indent=2))
else:
    print("Set RUN_TERRAIN_STAGE = True when you are ready to generate terrain outputs.")


In [ ]:
# Cell 5: Preview outputs after a successful run
import pandas as pd

if "metadata" not in globals():
    print("Run Cell 4 first to generate metadata and terrain outputs.")
else:
    csv_path = REPO_ROOT / metadata["output_files"]["attribute table (CSV)"]
    preview = pd.read_csv(csv_path)
    display(preview.head(10))
    print(f"Rows: {len(preview)}")
